In [5]:
%pip install openmeteo-requests
%pip install requests-cache retry-requests numpy pandas


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.

[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# spain data

In [ ]:
import os
import time
import requests
import pandas as pd
from datetime import date, timedelta

ARCHIVE_URL = "https://archive-api.open-meteo.com/v1/archive"
GEOCODE_URL = "https://geocoding-api.open-meteo.com/v1/search"

# Capitals (you can swap these)
CAPITALS = [
    {"country": "Spain",  "country_code": "ES", "city": "Madrid", "timezone": "Europe/Madrid"},
    {"country": "France", "country_code": "FR", "city": "Paris",  "timezone": "Europe/Paris"},
    {"country": "UK",     "country_code": "GB", "city": "London", "timezone": "Europe/London"},
]

# Choose your hourly variables (edit as needed)
HOURLY_VARS = [
    "temperature_2m",
    "relative_humidity_2m",
    "precipitation",
    "wind_speed_10m",
    "wind_direction_10m",
    "surface_pressure",
    "cloud_cover",
]

MODEL = "era5"

# last 10 years, ending a few days ago to avoid update-delay edges
today = date.today()
end_date = today - timedelta(days=6)
start_date = end_date.replace(year=end_date.year - 10)

OUT_DIR = "country_dfs"
os.makedirs(OUT_DIR, exist_ok=True)

def geocode(city: str, country_code: str) -> tuple[float, float]:
    params = {
        "name": city,
        "count": 1,
        "language": "en",
        "format": "json",
        "countryCode": country_code,
    }
    r = requests.get(GEOCODE_URL, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    results = js.get("results") or []
    if not results:
        raise ValueError(f"No geocoding results for {city} ({country_code})")
    return float(results[0]["latitude"]), float(results[0]["longitude"])

def fetch_hourly(lat: float, lon: float, tz: str, d0: date, d1: date) -> pd.DataFrame:
    params = {
        "latitude": lat,
        "longitude": lon,
        "start_date": d0.isoformat(),
        "end_date": d1.isoformat(),
        "hourly": ",".join(HOURLY_VARS),
        "timezone": tz,
        "models": MODEL,
        "timeformat": "iso8601",
    }
    r = requests.get(ARCHIVE_URL, params=params, timeout=120)
    r.raise_for_status()
    js = r.json()

    hourly = js.get("hourly") or {}
    times = hourly.get("time") or []
    if not times:
        return pd.DataFrame()

    df = pd.DataFrame({"time": pd.to_datetime(times)})
    for v in HOURLY_VARS:
        if v in hourly:
            df[v] = hourly[v]

    # metadata for merges
    df["latitude_used"] = js.get("latitude")
    df["longitude_used"] = js.get("longitude")
    df["timezone"] = js.get("timezone")
    return df

# Build one DF per country
dfs = {}

print(f"Fetching hourly from {start_date} to {end_date} (model={MODEL})")

for item in CAPITALS:
    country = item["country"]
    cc = item["country_code"]
    city = item["city"]
    tz = item["timezone"]

    lat, lon = geocode(city, cc)
    print(f"{country}: {city} -> ({lat:.4f}, {lon:.4f})")

    df = fetch_hourly(lat, lon, tz, start_date, end_date)

    df.insert(0, "country", country)
    df.insert(1, "country_code", cc)
    df.insert(2, "location", city)

    dfs[country.lower()] = df

    out_path = os.path.join(OUT_DIR, f"{country.lower()}_{city.lower()}_{start_date}_{end_date}_hourly.csv")
    df.to_csv(out_path, index=False)
    print(f"  saved: {out_path} | rows={len(df):,}")

    time.sleep(0.2)

# Convenience variables

df_spain = dfs["spain"]
df_france = dfs["france"]
df_uk = dfs["uk"]

print("\nDone. DataFrames available: df_spain, df_france, df_uk")

Fetching hourly from 2016-02-15 to 2026-02-15 (model=era5)
Spain: Madrid -> (40.4165, -3.7026)
  saved: country_dfs/spain_madrid_2016-02-15_2026-02-15_hourly.csv | rows=87,696
France: Paris -> (48.8534, 2.3488)
  saved: country_dfs/france_paris_2016-02-15_2026-02-15_hourly.csv | rows=87,696
UK: London -> (51.5085, -0.1257)
  saved: country_dfs/uk_london_2016-02-15_2026-02-15_hourly.csv | rows=87,696

Done. DataFrames available: df_spain, df_france, df_uk
